In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_25_try_0.pickle

trying: ['title_for_x_axis', 'title_for_x_axis_cell29', 'title_for_x_axis_cell22', 'title_for_x_axis_cell33', 'title_for_x_axis_cell20']
me:  6
me:  17
me:  10
me:  21
me:  8
trying: ['title_for_x_axis', 'title_for_x_axis_cell29', 'title_for_x_axis_cell22', 'title_for_x_axis_cell33', 'title_for_x_axis_cell20']
me:  6
me:  17
me:  10
me:  21
me:  8
trying: ['responses_df_2022', 'responses_df_2022_cell10']
me:  16
me:  16
trying: ['responses_df_2022', 'responses_df_2022_cell10']
me:  16
me:  16
trying: ['title_for_x_axis', 'title_for_x_axis_cell29', 'title_for_x_axis_cell22', 'title_for_x_axis_cell33', 'title_for_x_axis_cell20']
me:  6
me:  17
me:  10
me:  21
me:  8
trying: ['title_for_x_axis', 'title_for_x_axis_cell29', 'title_for_x_axis_cell22', 'title_for_x_axis_cell33', 'title_for_x_axis_cell20']
me:  6
me:  17
me:  10
me:  21
me:  8
trying: ['count_then_return_percent_23']
me:  11
trying: ['percentages_cell23']
me:  11
trying: ['question_of_interest_cell23']
me:  11
trying: ['sns']


me:  0
trying: ['glob']
me:  0
trying: ['pd']
me:  0
trying: ['orientation_for_chart']
me:  5
trying: ['count_then_return_percent_24']
me:  12
trying: ['percentages_cell24']
me:  12
trying: ['question_of_interest_cell22']
me:  10
trying: ['title_for_chart_cell24']
me:  12
trying: ['add_year_column_to_dataframes_22']
me:  10
trying: ['title_for_y_axis_cell24']
me:  12
trying: ['responses_df_2018']
me:  1
trying: ['replace_hyphen_with_en_dash']
me:  2
trying: ['question_of_interest_cell24']
me:  12
trying: ['India_responses_df_2022']
me:  12
trying: ['responses_df_2019']
me:  1
trying: ['bar_chart_multiple_choice_24']
me:  12
trying: ['USA_responses_df_2022']
me:  12
trying: ['percentages']
me:  5
trying: ['cols_jupyter']
me:  25
trying: ['cols_vs']
me:  25
trying: ['subset_df']
me:  25
trying: ['age_df_combined']
me:  8
trying: ['j_mask']
me:  25
trying: ['responses_in_order']
me:  5
trying: ['age_df_combined_cell22']
me:  10
trying: ['question_of_interest']
me:  5
trying: ['count_then_

me:  17
trying: ['responses_df_2018_cell10']
me:  21
trying: ['subset_of_degrees']
me:  17
trying: ['languages']
me:  23
trying: ['grab_subset_of_data_35']
me:  23
trying: ['language_df_combined_counts']
me:  23
trying: ['language_df_combined_percentages']
me:  23
trying: ['df_cell35']
me:  23
trying: ['convert_df_of_counts_to_percentages_35']
me:  23
trying: ['language_df_combined']
me:  23
trying: ['combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35']
me:  23
trying: ['question_of_interest_cell37']
me:  25
trying: ['responses_in_order_cell21']
me:  9
trying: ['question_of_interest_cell35']
me:  23
trying: ['percentages_cell21']
me:  9
trying: ['count_then_return_percent_21']
me:  9
trying: ['question_of_interest_cell21']
me:  9
trying: ['ide_df_2022_cell37']
me:  25
trying: ['question_of_interest_cell19']
me:  7
trying: ['count_then_return_percent_19']
me:  7
trying: ['percentages_cell19']
me:  7
trying: ['warnings']
me:  0
trying: ['np']
me

In [3]:
%%RecordEvent
%%time
### cell 26 ###

def grab_subset_of_data_38(original_df, question_of_interest):
    # Select only columns containing the question, rename by suffix after '-' and drop all‐NaN rows
    subset = original_df.filter(like=question_of_interest, axis=1)
    if subset.empty:
        return subset
    new_names = subset.columns.str.split('-').str[-1].str.lstrip()
    subset.columns = new_names
    return subset.dropna(how='all')


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_38(
        question_of_interest, include_2017=None):
    # Map years to their DataFrames
    year_sources = [
        ('2022', responses_df_2022_cell10),
        ('2021', responses_df_2021),
        ('2020', responses_df_2020),
        ('2019', responses_df_2019_cell10),
        ('2018', responses_df_2018_cell10),
    ]
    if include_2017 is not None:
        year_sources.insert(0, ('2017', responses_df_2017))
    # Grab subsets and tag with year
    dfs = []
    for year, df in year_sources:
        sub = grab_subset_of_data_38(df, question_of_interest)
        if not sub.empty:
            dfs.append(sub.assign(year=year))
    if not dfs:
        return pd.DataFrame(), pd.DataFrame()
    combined = pd.concat(dfs, ignore_index=True)
    counts = combined.groupby('year').count().reset_index()
    return combined, counts


def convert_df_of_counts_to_percentages_38(df, df_counts):
    # Compute respondent counts per year and divide
    denom = df['year'].value_counts()
    pct = df_counts.set_index('year').div(denom, axis=0).mul(100).reset_index()
    return pct

# Normalize 2018 column names in one pass
correct_phrasing = 'Jupyter Notebook / JupyterLab'
question_of_interest_cell38 = (
    "Which of the following integrated development environments (IDE's) do you use on a regular basis?"
)
alternate_phrasing = (
    "Which of the following integrated development environments (IDE's) have you used at work or school in the last 5 years?"
)
responses_df_2018_cell10.columns = (
    responses_df_2018_cell10.columns
        .str.replace('Jupyter/IPython', correct_phrasing, regex=False)
        .str.replace(alternate_phrasing, question_of_interest_cell38, regex=False)
)

x_axis_title = 'Percentage of respondents'
ide_df_combined, ide_df_combined_counts = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_38(
        question_of_interest_cell38
    )
)

# Define how to merge variant columns into single indicators
mapping = {
    'Jupyter Notebook / JupyterLab': [
        'Jupyter Notebook', 'JupyterLab ',
        'Jupyter (JupyterLab, Jupyter Notebooks, etc) ',
        'Jupyter Notebook / JupyterLab'
    ],
    'Visual Studio / Visual Studio Code (VSCode)': [
        'Visual Studio Code', 'Visual Studio',
        'Visual Studio / Visual Studio Code ', 'Visual Studio ',
        'Visual Studio Code (VSCode) ', 'Click to write Choice 13'
    ],
    'RStudio': ['RStudio', 'RStudio '],
    'PyCharm': ['PyCharm ', 'PyCharm'],
    'MATLAB': ['MATLAB', 'MATLAB '],
}

# Build cleaned DataFrame with one column per choice
ide_df_clean = ide_df_combined.copy()
for target, variants in mapping.items():
    mask = ide_df_clean[variants].notna().any(axis=1)
    ide_df_clean[target] = target
    ide_df_clean.loc[~mask, target] = np.nan
# Drop all raw variant columns except our new columns
raw_to_drop = set(v for variants in mapping.values() for v in variants) - set(mapping.keys())
ide_df_clean.drop(columns=list(raw_to_drop), inplace=True)

# Recompute counts & percentages
ide_df_counts2 = ide_df_clean.groupby('year').count().reset_index()
ide_df_percentages = convert_df_of_counts_to_percentages_38(ide_df_clean, ide_df_counts2)
# Keep only the five choices in a consistent order
cols_order = [
    'Jupyter Notebook / JupyterLab',
    'Visual Studio / Visual Studio Code (VSCode)',
    'RStudio',
    'PyCharm',
    'MATLAB'
]
ide_df_percentages_cell38 = ide_df_percentages[['year'] + cols_order]

# Melt, sort, and prepare for plotting
df_cell38 = (
    ide_df_percentages_cell38
        .melt(id_vars=['year'], value_vars=cols_order, var_name='', value_name='value')
        .sort_values(['year', 'value'])
        .reset_index(drop=True)
)
df_cell38.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   year    25 non-null     object 
 1           25 non-null     object 
 2   value   25 non-null     float64
dtypes: float64(1), object(2)
memory usage: 728.0+ bytes
CPU times: user 75.9 ms, sys: 0 ns, total: 75.9 ms
Wall time: 75.6 ms


In [4]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_26_try_0.pickle

migration speed (bps): 729489533.6678513


---------------------------
variables to migrate:
grab_subset_of_data_27 144
question_of_interest_cell27 153
df 5569
count_then_return_percent_28 144
percentages_cell28 561
responses_df_2019_cell10 15975586
responses_df_2022 25378603
responses_df_2020 25517727
create_dataframe_of_counts_16 144
percentages_per_country_df 4474
responses_df_2021 34255607
question_of_interest_cell28 160
responses_in_order_cell28 152
responses_df_2017 15719078
convert_df_of_counts_to_percentages_35 144
language_df_combined 8514579
load_survey_data 144
combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35 144
factor 24
question_of_interest_cell35 106
directory 163
languages 120
file_name_2018 76
grab_subset_of_data_35 144
base_dir_2022 155
question_of_interest_cell20 76
language_df_combined_counts 1409
file_name_2019 78
combine_subset_of_data_from_multiple_years_20 144
language_df_combined_percentages 1144
file_name_2022 81
count_then_return_percent_20 144
df_cell35 34

In [5]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['responses_df_2022', 'responses_df_2020', 'responses_df_2021', 'responses_df_2018', 'responses_df_2019']
Intermediate variables ['base_dir_2021', 'directory', 'factor', 'base_dir_2018', 'file_name_2017', 'base_dir_2020', 'base_dir_2022', 'base_dir_2019', 'base_dir_2017', 'file_name_2022', 'directory_cell8', 'file_name_2021', 'responses_df_2017', 'file_name_2018', 'file_name_2020', 'file_name_2019']
Future variables ['responses_df_2022_cell10', 'percentages', 'responses_df_2018_cell10', 'responses_df_2017']
Modified dataframes
======= Cell 2 =======
Input variables ['responses_df_2018', 'responses_df_2022', 'responses_df_2019']
Active variables ['responses_df_2022_cell10', 'responses_df_2019_cell10', 'responses_df_2022']
Intermediate variables ['responses_df_2018_cell10']
Future variables ['responses_d

In [6]:

with open("/home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_26_try_0.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[26], f)


In [7]:
opt_output = Out.get(4)

In [ ]:
assert False, 'x_axis_title is incorrectly modified in the optimized code.'